# Nuevo Entrenamiento YOLO26 - Dataset Crudo (V2)

Este notebook está optimizado para mejorar el rendimiento del modelo YOLO a utilizar. Para esto, vamos a generar el nuevo `dataset_mixto_crudo` que será finalmente utilizado en la siguiente notebook `04_1_documentacion_entrenamiento.ipynb` para entrenar el modelo de visión computacional, donde para cuyo dataset vamos a utilizar:
1. **Imágenes Crudas:** Sin redimensión destructiva ni aumentos estáticos de Roboflow.
2. **Inyección de imágenes del dataset global:** Se incluirán solo imágenes con etiquetas y un pequeño volumen de imágenes background.

## 1. Configuración de Entorno (Colab & Drive)

In [ ]:
from google.colab import drive
import os

# 1. Conexión directa a Drive
drive.mount("/content/drive")

# 2. Instalación de librerías
!pip install -q ultralytics roboflow

# 3. Definición de rutas para Colab
ruta_mixta = "/content/drive/MyDrive/YOLO_Entrenamientos/dataset_mixto_crudo"
ruta_dataset_mixto_yaml = os.path.join(ruta_mixta, "dataset.yaml")
output_project = (
    "/content/drive/MyDrive/YOLO_Entrenamientos/Training_dataset_mixto_crudo"
)
nombre_entrenamiento = "YOLO26M_Mixto_Crudo_V1"

print(f"--- Configuración de rutas completada ---")
print(f"Directorio del dataset: {ruta_mixta}")
print(f"Archivo YAML: {ruta_dataset_mixto_yaml}")

## 2. Descarga de Datos (Roboflow)

In [ ]:
from roboflow import Roboflow
from dotenv import load_dotenv
import os

# --- CARGAR VARIABLES DE ENTORNO ---
load_dotenv()  # Carga desde .env si existe
api_key = os.getenv("ROBOFLOW_API_KEY")

if not api_key:
    raise ValueError(
        "❌ ROBOFLOW_API_KEY no encontrada en variables de entorno. Asegúrate de configurarla en Colab Secrets o en un archivo .env"
    )

# --- DESCARGA DE ROBOFLOW ---
rf = Roboflow(api_key=api_key)
project = rf.workspace("pics-workspace").project("deteccion-vial-moreno")
version = project.version(2)
dataset = version.download("yolo26")

print(f"✅ Dataset descargado en sesión temporal: {dataset.location}")

## 3. Ensamblaje del Dataset Mixto Crudo
*(Ejecutar solo si el dataset_mixto_crudo no existe aún en Drive)*

### 3.1. Fuentes de Datos

#### Moreno (Local / Roboflow)
* **Origen:** Dataset específico de la zona de Moreno descargado desde Roboflow (versión cruda/raw).
* **Tratamiento:** Se importa de manera íntegra (100% de imágenes y etiquetas). Es el núcleo del entrenamiento.

#### Global (Inyección Externa)
* **Origen:** Dataset global de imágenes de calles - RDD2022.
* **Tratamiento:** Se utiliza para aumentar la variedad de ejemplos, pero con filtros estrictos.

### 3.2. Reglas de Filtrado y Balanceo

Para evitar que el modelo se confunda o ignore los baches reales, se aplican las siguientes reglas al inyectar datos del dataset Global:

1. **Prioridad de Etiquetas (Útiles):** Se inyectan **todas** las imágenes del dataset global que contienen etiquetas (`D20`, `D40`). Si tiene un bache marcado, entra al dataset mixto.
2. **Control de Background (Fondo):** Las imágenes sin etiquetas (background) son útiles para reducir falsos positivos, pero en exceso pueden degradar la precisión. Se limitan de forma aleatoria:
   * **Train:** Máximo 400 imágenes.
   * **Val:** Máximo 100 imágenes.
   * **Test:** 0 imágenes.
3. **Prefijo de Seguridad (`g_`):** Todas las imágenes y etiquetas provenientes del dataset global se renombran con el prefijo `g_` (ej. `g_Czech_0001.jpg`). Esto evita que archivos con el mismo nombre (ej. `1.jpg`) se sobrescriban entre los distintos datasets.

In [ ]:
import shutil
import random

# --- RUTA GLOBAL (Asegúrate de que este ZIP/RAR esté en tu Drive o súbelo a Colab) ---
ruta_global_base = "/content/drive/MyDrive/dataset_global"
ruta_local_cruda = dataset.location

if not os.path.exists(ruta_mixta):
    print("⏳ Creando estructura de dataset mixto en Drive...")
    # 1. Creamos las carpetas de imágenes y etiquetas para cada split (train, val, test)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(ruta_mixta, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(ruta_mixta, split, "labels"), exist_ok=True)

    print("⏳ Copiando Moreno Crudo...")
    # 2. Migramos los datos descargados de Roboflow a su ubicación final en Drive
    for split_rf, split_dest in [
        ("train", "train"),
        ("valid", "val"),
        ("test", "test"),
    ]:
        img_dir = os.path.join(ruta_local_cruda, split_rf, "images")
        lbl_dir = os.path.join(ruta_local_cruda, split_rf, "labels")
        if os.path.exists(img_dir):
            for fn in os.listdir(img_dir):
                shutil.copy(
                    os.path.join(img_dir, fn),
                    os.path.join(ruta_mixta, split_dest, "images", fn),
                )
                lbl = os.path.splitext(fn)[0] + ".txt"
                src_lbl = os.path.join(lbl_dir, lbl)
                if os.path.exists(src_lbl):
                    shutil.copy(
                        src_lbl, os.path.join(ruta_mixta, split_dest, "labels", lbl)
                    )

    print("⏳ Inyectando Global y Background...")
    # 3. Definimos límites para imágenes de fondo (background) del dataset global
    CANTIDAD_BG = {"train": 400, "val": 100, "test": 0}

    def inyectar_global(split_dest, num_bg):
        # Buscamos imágenes en el dataset global ya filtrado previamente
        src_img = os.path.join(
            ruta_global_base, "valid" if split_dest == "val" else split_dest, "images"
        )
        src_lbl = os.path.join(
            ruta_global_base, "valid" if split_dest == "val" else split_dest, "labels"
        )
        if not os.path.exists(src_img):
            return

        # Clasificamos entre imágenes con baches (útiles) y sin etiquetas (vacías/background)
        utiles, vacias = [], []
        for fn in os.listdir(src_img):
            l_path = os.path.join(src_lbl, os.path.splitext(fn)[0] + ".txt")
            if os.path.exists(l_path) and os.path.getsize(l_path) > 0:
                utiles.append(fn)
            else:
                vacias.append(fn)

        # Copiamos todas las útiles con prefijo 'g_' para evitar duplicados
        for fn in utiles:
            shutil.copy(
                os.path.join(src_img, fn),
                os.path.join(ruta_mixta, split_dest, "images", f"g_{fn}"),
            )
            shutil.copy(
                os.path.join(src_lbl, os.path.splitext(fn)[0] + ".txt"),
                os.path.join(
                    ruta_mixta, split_dest, "labels", f"g_{os.path.splitext(fn)[0]}.txt"
                ),
            )

        # Inyectamos una muestra aleatoria de imágenes de fondo (background)
        if vacias and num_bg > 0:
            for fn in random.sample(vacias, min(num_bg, len(vacias))):
                shutil.copy(
                    os.path.join(src_img, fn),
                    os.path.join(ruta_mixta, split_dest, "images", f"g_{fn}"),
                )

    # 4. Ejecutamos la inyección para entrenamiento y validación
    for s in ["train", "val"]:
        inyectar_global(s, CANTIDAD_BG[s])

    # 5. Creamos el archivo de configuración YAML necesario para YOLO
    yaml_content = f"path: {ruta_mixta}\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 3\nnames: ['D20', 'D40', 'calle_tierra']"
    with open(ruta_dataset_mixto_yaml, "w") as f:
        f.write(yaml_content)
    print("✅ Dataset Mixto ensamblado en Drive.")
else:
    print("✅ El dataset mixto ya existe en Drive. Saltando ensamblaje.")